# WER · CER 이해하기

음성 인식(STT) 결과가 **정답 문장과 얼마나 다른지**를 숫자로 나타내는 방법을 처음부터 직접 만들어 봅니다.

- 8강(WER·CER와 오류 분석)은 수업에서 다루지 않아서, 프로젝트(`scoring.py`)에 쓰기 위해 **자습으로 정리**한 노트북입니다.
- 이 노트북의 코드는 모두 직접 실행해서 결과를 확인했습니다. 확인하지 못한 내용은 **[추정]** 또는 **[일반 지식]** 으로 표시했습니다.
- 사용 패키지는 `pandas`와 표준 라이브러리(`re`, `unicodedata`, `difflib`, `random`)뿐이라 `voice-ai` 환경에서 바로 실행됩니다.

| 순서 | 내용 |
|---|---|
| 1 | WER·CER가 무엇을 재는가 (치환·삭제·삽입) |
| 2 | 편집 거리(Levenshtein) 표를 직접 채우기 |
| 3 | 역추적으로 "어디가 틀렸는지" 찾기 |
| 4 | WER·CER 함수 만들고 손계산과 맞춰 보기 |
| 5 | WER가 1을 넘는 경우 |
| 6 | 단어 단위 vs 글자 단위 (영어·한국어) |
| 7 | 정규화가 점수를 바꾼다 |
| 8 | `difflib`와 편집 거리는 다를 수 있다 |
| 9 | 연습 문제 |
| 10 | 여러 문장의 WER/CER 합치기 (Micro·Macro) |
| 11 | 테스트와 경계 조건 |
| 12 | 프로젝트 `scoring.py`와의 차이 |
| 13 | 정리와 프로젝트 연결 |

> **2026-11-02 보완**: 외부 검토 의견을 받아 코드로 하나씩 확인한 뒤, 기존 셀은 그대로 두고 `(보완)` 표시가 붙은 셀을 끼워 넣었습니다. 확인해 보니 의견과 다르게 나온 부분(프로젝트 코드의 공백 처리, 따옴표 처리)은 해당 셀에 **[실행 확인]** 으로 적었습니다. DP 표가 채워지는 과정은 `experiments/WER, CER/DP_편집거리_이해하기.html`에서 단계별로 볼 수 있습니다.

## 1. WER · CER가 무엇을 재는가

STT가 이렇게 인식했다고 해 봅시다.

| | 문장 |
|---|---|
| 정답(reference) | the cat sat on the mat |
| 인식(hypothesis) | the cat sit on mat |

사람이 보면 틀린 곳이 두 군데입니다.

- `sat` → `sit` : 다른 단어로 **바뀜** → **치환(Substitution, S)**
- `the` 하나가 **빠짐** → **삭제(Deletion, D)**
- (인식 쪽에 정답에 없는 단어가 **끼어들면** → **삽입(Insertion, I)**)

$$\text{WER} = \frac{S + D + I}{N}$$

- $N$ = **정답 문장의 단어 수**(인식 문장의 단어 수가 아님)
- 위 예: $S=1,\ D=1,\ I=0,\ N=6$ 이므로 WER $= 2/6 \approx 0.333$

**CER(Character Error Rate)** 은 같은 식을 **단어 대신 글자** 단위로 계산한 것입니다.

### 1-2. WER는 "단어 개수 비교"가 아니라 "순서가 있는 단어 열의 정렬"입니다 (보완)

WER는 정답과 인식 문장의 단어 **개수**만 견주는 지표가 아닙니다. 두 단어 열을 **순서대로** 맞춰 보고, 정답을 인식 결과로 바꾸는 데 필요한 **최소 편집 횟수**를 셉니다.

| 구분 | 문장 |
|---|---|
| 정답(reference) | I really like this song |
| 인식(hypothesis) | I like this song |

- 단어 수만 비교하면 "5개와 4개"라는 사실밖에 모릅니다. 실제로는 `really`가 **삭제**된 것이므로 $D=1,\ N=5$ → WER $= 1/5 = 0.2$ 입니다.
- 순서도 평가 대상입니다. 같은 단어가 들어 있어도 순서가 바뀌면 편집이 필요합니다. 예를 들어 `the cat sat` 를 `sat the cat` 으로 읽으면 단어 집합은 같지만 WER는 0이 아닙니다(4절 끝의 확인 셀에서 계산합니다).
- 그래서 WER는 단어의 **집합(set)** 이 아니라 **순서가 있는 열(sequence)** 을 비교하는 지표입니다.

## 2. 편집 거리(Levenshtein distance) 표 채우기

"정답을 인식 결과로 바꾸려면 **치환·삭제·삽입을 최소 몇 번** 해야 하는가"가 편집 거리이고, 이 최소 횟수가 곧 $S+D+I$ 입니다.

표 `d[i][j]`의 뜻: **정답의 앞 `i`개와 인식의 앞 `j`개를 맞추는 데 필요한 최소 횟수**

- 첫 행·첫 열: 한쪽이 비어 있으면 상대 길이만큼 삭제(또는 삽입)해야 하므로 `d[i][0] = i`, `d[0][j] = j`
- 나머지 칸: 아래 세 값 중 **최솟값**
  - 대각선 `d[i-1][j-1]` + (두 토큰이 같으면 0, 다르면 1) → 일치/치환
  - 위쪽 `d[i-1][j]` + 1 → 정답 토큰을 지움(삭제)
  - 왼쪽 `d[i][j-1]` + 1 → 인식 토큰을 끼워 넣음(삽입)

In [1]:
import pandas as pd

def edit_matrix(ref, hyp):
    """ref, hyp: 토큰 리스트(단어 리스트 또는 글자 리스트). 편집 거리 표(2차원 리스트)를 돌려줌."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i                      # 정답 i개를 모두 지워야 함
    for j in range(m + 1):
        d[0][j] = j                      # 인식 j개를 모두 끼워 넣어야 함
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,          # 삭제
                          d[i][j - 1] + 1,          # 삽입
                          d[i - 1][j - 1] + cost)   # 일치 또는 치환
    return d

def show_matrix(ref, hyp):
    d = edit_matrix(ref, hyp)
    return pd.DataFrame(d, index=["(시작)"] + ref, columns=["(시작)"] + hyp)

ref = "the cat sat on the mat".split()
hyp = "the cat sit on mat".split()
show_matrix(ref, hyp)

,(시작),the,cat,sit,on,mat
(시작),0,1,2,3,4,5
the,1,0,1,2,3,4
cat,2,1,0,1,2,3
sat,3,2,1,1,2,3
on,4,3,2,2,1,2
the,5,4,3,3,2,2
mat,6,5,4,4,3,2


**표 읽는 법**

- 오른쪽 아래 칸(`mat` 행, `mat` 열)의 값이 **전체 편집 거리**입니다. 위 표에서는 **2** 입니다.
- 대각선으로 내려가며 값이 그대로면 일치, 1 늘어나면 치환입니다. 위쪽에서 내려오며 1 늘면 삭제, 왼쪽에서 오며 1 늘면 삽입입니다.
- 이 표만으로는 **거리(몇 번 틀렸나)** 는 알지만 **어디가 어떻게 틀렸나**는 모릅니다. 다음 절에서 표를 거꾸로 따라가 알아냅니다.

### 2-2. 왜 "표(DP)"로 푸는가 — 같은 계산을 반복하지 않기 위해서 (보완)

`d[i][j]`에는 **정답의 앞 `i`개와 인식의 앞 `j`개를 맞추는 최소 횟수**가 들어 있습니다. 칸 하나가 "작은 문제 하나의 정답"이고, 큰 칸은 이미 채워진 **위·왼쪽·대각선 왼쪽 위** 세 칸만 보고 계산합니다. 이 방식을 **동적 계획법(Dynamic Programming)** 이라고 합니다 **[일반 지식]**.

DP가 성립하는 두 가지 조건입니다 **[일반 지식]**.

- **최적 부분 구조**: 큰 문제의 최적해가 작은 문제의 최적해로 만들어집니다(`d[i][j]`는 `d[i-1][j]`, `d[i][j-1]`, `d[i-1][j-1]`의 최적해에서 나옵니다).
- **중복되는 부분 문제**: 같은 작은 문제를 여러 경로가 반복해서 필요로 합니다. 표에 한 번만 적어 두면 다시 계산하지 않아도 됩니다.

복잡도는 정답 토큰 수를 $N$, 인식 토큰 수를 $M$이라고 할 때 시간·공간 모두 $O(N \times M)$ 입니다(지금 구현은 표 전체를 보관). 한 문장 정도의 길이에서는 충분히 빠릅니다.

아래 셀은 **표 없이 재귀만으로** 같은 값을 구했을 때 같은 칸을 몇 번이나 다시 계산하는지 센 것입니다.

In [2]:
# 표(메모) 없이 재귀로만 푼 버전: f(i, j) = 정답 앞 i개 vs 인식 앞 j개의 최소 수정 횟수
def edit_distance_naive(ref_tokens, hyp_tokens):
    calls = 0
    def f(i, j):
        nonlocal calls            # 바깥 함수의 변수 calls를 안쪽 함수에서 고쳐 쓰겠다는 선언
        calls += 1
        if i == 0:
            return j
        if j == 0:
            return i
        cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
        return min(f(i - 1, j) + 1, f(i, j - 1) + 1, f(i - 1, j - 1) + cost)
    return f(len(ref_tokens), len(hyp_tokens)), calls

dist_naive, calls = edit_distance_naive(ref, hyp)
cells = (len(ref) + 1) * (len(hyp) + 1)
print(f"편집 거리 {dist_naive} (표로 구한 값 {edit_matrix(ref, hyp)[-1][-1]})  |  재귀 호출 {calls}번  vs  DP 표 칸 {cells}개")

편집 거리 2 (표로 구한 값 2)  |  재귀 호출 5479번  vs  DP 표 칸 42개


같은 값(2)을 얻는 데 재귀는 수천 번, 표는 42칸입니다. 문장이 길어질수록 이 차이는 급격히 벌어집니다.

> DP 표가 채워지는 과정을 칸 단위로 눈으로 따라가고 싶다면 `experiments/WER, CER/DP_편집거리_이해하기.html`을 브라우저로 여세요(단계별로 넘겨 보는 표, 역추적 애니메이션, 직접 채워 보는 연습 모드가 있습니다).

## 3. 역추적(backtracking) — 어디가 틀렸는지 찾기

표의 오른쪽 아래에서 출발해 "이 값이 어느 칸에서 왔는가"를 거꾸로 따라가면 치환·삭제·삽입 목록을 얻습니다.

- 같은 거리를 만드는 경로가 **여러 개**일 수 있습니다. 어느 것을 고를지는 `if` 순서(여기서는 일치/치환 → 삭제 → 삽입)로 정해집니다. 거리(점수)는 같고, **오류의 위치 표시가 달라질 수 있습니다** **[추정: 코드의 순서 규칙에서 나온 해석]**.

In [3]:
def backtrace(ref, hyp, d):
    i, j = len(ref), len(hyp)
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1] and d[i][j] == d[i - 1][j - 1]:
            ops.append(("일치", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + 1:
            ops.append(("치환", ref[i - 1], hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(("삭제", ref[i - 1], None)); i -= 1
        else:
            ops.append(("삽입", None, hyp[j - 1])); j -= 1
    return ops[::-1]

d = edit_matrix(ref, hyp)
ops = backtrace(ref, hyp, d)
pd.DataFrame(ops, columns=["종류", "정답 쪽", "인식 쪽"])

,종류,정답 쪽,인식 쪽
0,일치,the,the
1,일치,cat,cat
2,치환,sat,sit
3,일치,on,on
4,삭제,the,NaN
5,일치,mat,mat


위 표에서 `치환`이 1개(`sat` → `sit`), `삭제`가 1개(`the`)입니다. 이 개수를 세면 $S, D, I$ 가 나옵니다.

### 3-2. 최적 경로가 여러 개일 수 있다 — 거리는 같아도 S·D·I와 표시가 달라집니다 (보완)

표의 마지막 값(편집 거리)은 하나로 정해지지만, 그 값을 만드는 **경로는 여러 개**일 수 있습니다. 위 `backtrace()`는 그중 하나를 `if` 순서로 고를 뿐이고, 그것이 유일한 정답 경로라는 뜻은 아닙니다.

- **오류율**(거리 ÷ N)은 어느 경로를 골라도 같습니다.
- 하지만 **S·D·I의 개수 구성**과 **화면에 표시되는 틀린 위치**는 경로에 따라 달라질 수 있습니다.

아래 셀은 표를 따라 가능한 **모든** 최적 경로를 모아 보여 줍니다(짧은 예시에서만 사용: 경로 수가 빠르게 늘어납니다 **[일반 지식]**).

In [4]:
def all_optimal_paths(ref_tokens, hyp_tokens):
    # 표 d를 따라, 최소 거리를 만드는 모든 경로를 (종류, 정답 쪽, 인식 쪽) 목록으로 모은다(학습용).
    d = edit_matrix(ref_tokens, hyp_tokens)
    def walk(i, j):
        if i == 0 and j == 0:
            yield []
            return
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (ref_tokens[i - 1] != hyp_tokens[j - 1]):
            kind = "일치" if ref_tokens[i - 1] == hyp_tokens[j - 1] else "치환"
            for p in walk(i - 1, j - 1):
                yield p + [(kind, ref_tokens[i - 1], hyp_tokens[j - 1])]
        if i > 0 and d[i][j] == d[i - 1][j] + 1:
            for p in walk(i - 1, j):
                yield p + [("삭제", ref_tokens[i - 1], None)]
        if j > 0 and d[i][j] == d[i][j - 1] + 1:
            for p in walk(i, j - 1):
                yield p + [("삽입", None, hyp_tokens[j - 1])]
    return list(walk(len(ref_tokens), len(hyp_tokens)))

def show_path(ops):
    parts = []
    for kind, a, b in ops:
        parts.append(a if kind == "일치" else f"[{a}→{b}]" if kind == "치환" else f"[-{a}-]" if kind == "삭제" else f"[+{b}+]")
    return " ".join(parts)

def count_sdi(ops):
    return "S={} D={} I={}".format(*(sum(o[0] == k for o in ops) for k in ("치환", "삭제", "삽입")))

for a, b in [("a b", "b a"), ("we'll sing together", "we are seeing together")]:
    ra, hb = a.split(), b.split()
    print(f"정답 {a!r}  /  인식 {b!r}  (편집 거리 {edit_matrix(ra, hb)[-1][-1]})")
    for ops in all_optimal_paths(ra, hb):
        print("   ", show_path(ops).ljust(46), count_sdi(ops))
    print("    이 노트북의 backtrace()가 고른 것:", show_path(backtrace(ra, hb, edit_matrix(ra, hb))))
    print()

정답 'a b'  /  인식 'b a'  (편집 거리 2)
    [a→b] [b→a]                                    S=2 D=0 I=0
    [+b+] a [-b-]                                  S=0 D=1 I=1
    [-a-] b [+a+]                                  S=0 D=1 I=1
    이 노트북의 backtrace()가 고른 것: [a→b] [b→a]

정답 "we'll sing together"  /  인식 'we are seeing together'  (편집 거리 3)
    [+we+] [we'll→are] [sing→seeing] together      S=2 D=0 I=1
    [we'll→we] [+are+] [sing→seeing] together      S=2 D=0 I=1
    [we'll→we] [sing→are] [+seeing+] together      S=2 D=0 I=1
    이 노트북의 backtrace()가 고른 것: [+we+] [we'll→are] [sing→seeing] together



- `a b` / `b a` 는 "치환 2번"과 "삭제 1 + 삽입 1" 이 **같은 거리 2**입니다. 오류율은 같지만 S·D·I 구성이 다릅니다.
- `we'll sing together` / `we are seeing together` 는 이 노트북의 `backtrace()`가 `[+we+] [we'll→are] ...`처럼 **서로 안 닮은 단어끼리** 짝지은 경로를 고릅니다. 사람이 보기에는 `we'll`↔`we`, `sing`↔`seeing`을 짝짓는 경로가 자연스럽습니다.
- 프로젝트의 `scoring.py`는 이 문제를 `_tie_scores()`로 풉니다. **거리가 같은 경로들 중** 닮은 단어끼리 짝지은 정도의 합이 가장 큰 경로를 고릅니다. 이 규칙은 **오류율을 바꾸지 않고 표시만 바꾸는** 보조 규칙이며, 이 노트북에는 없는 프로젝트의 추가 규칙입니다(12절에서 나란히 비교).

## 4. WER · CER 함수 만들기

- 입력은 **이미 토큰으로 나눈 리스트**로 받고, 문장 → 토큰은 따로 함수로 만듭니다(WER는 단어, CER는 글자).
- 정답이 비어 있으면 분모 $N=0$ 이라 계산할 수 없으므로 **오류로 처리**합니다.

In [5]:
def score(ref_tokens, hyp_tokens):
    if len(ref_tokens) == 0:
        raise ValueError("정답이 비어 있어 오류율을 계산할 수 없습니다 (분모 N = 0)")
    d = edit_matrix(ref_tokens, hyp_tokens)
    ops = backtrace(ref_tokens, hyp_tokens, d)
    S = sum(o[0] == "치환" for o in ops)
    D = sum(o[0] == "삭제" for o in ops)
    I = sum(o[0] == "삽입" for o in ops)
    N = len(ref_tokens)
    distance = d[-1][-1]
    assert S + D + I == distance          # 검산: 역추적한 오류 수의 합 = 편집 거리
    return {"S": S, "D": D, "I": I, "N": N, "거리": distance, "오류율": distance / N, "ops": ops}

def wer(ref, hyp):
    return score(ref.split(), hyp.split())

def cer(ref, hyp):
    # 이 노트북의 선택: 공백을 빼고 글자 단위로 비교 (공백을 글자로 세는 방식도 있음 [일반 지식])
    return score(list(ref.replace(" ", "")), list(hyp.replace(" ", "")))

ref_s = "the cat sat on the mat"
hyp_s = "the cat sit on mat"
for name, f in [("WER", wer), ("CER", cer)]:
    r = f(ref_s, hyp_s)
    print(f"{name}: S={r['S']} D={r['D']} I={r['I']} N={r['N']}  →  ({r['S']}+{r['D']}+{r['I']})/{r['N']} = {r['오류율']:.3f}")

WER: S=1 D=1 I=0 N=6  →  (1+1+0)/6 = 0.333
CER: S=1 D=3 I=0 N=17  →  (1+3+0)/17 = 0.235


**손계산과 맞추기** — 공백을 뺀 글자열은 정답 `thecatsatonthemat`(17자), 인식 `thecatsitonmat`(14자)입니다. `sat`→`sit`에서 글자 1개 치환, `the` 3글자 삭제이므로 $S=1,\ D=3,\ I=0$ → $(1+3)/17 \approx 0.235$ 가 되어야 하고, 위 출력과 같은지 확인합니다(같은 거리의 다른 경로가 있어도 **합은 같습니다**).

In [6]:
assert wer(ref_s, hyp_s)["거리"] == 2 and abs(wer(ref_s, hyp_s)["오류율"] - 2/6) < 1e-9
assert cer(ref_s, hyp_s)["거리"] == 4 and abs(cer(ref_s, hyp_s)["오류율"] - 4/17) < 1e-9
print("손계산과 일치: WER 2/6, CER 4/17")

손계산과 일치: WER 2/6, CER 4/17


### 4-2. CER 보완: 공백 문자와 유니코드 정규화 (보완)

위 `cer()`는 `ref.replace(" ", "")`로 **ASCII 공백(U+0020)만** 지웁니다. 실제 문자열에는 탭, 줄바꿈, 줄 바꿈 없는 공백(NBSP, U+00A0) 같은 공백도 있을 수 있습니다. 또 `list(text)`는 문자열을 **유니코드 코드 포인트** 단위로 쪼개는데, 화면에서 같은 글자도 코드 포인트 구성이 다를 수 있습니다.

- `가` = 한 코드 포인트(U+AC00, NFC 형태)
- `가` = 두 코드 포인트(U+1100 + U+1161, NFD 형태)

정규화 없이 비교하면 눈에 같은 글자가 **서로 다른 글자**로 계산됩니다. 그래서 CER는 계산 전에 **NFC로 맞추고**, **모든 공백 문자를 제거**하는 것이 일관적입니다. 공백을 무시하는 것은 이 프로젝트의 **평가 정책**이며, 공백을 글자로 세는 CER와는 값이 다를 수 있습니다 **[일반 지식]**.

In [7]:
import unicodedata

cer_v1 = cer          # 4절의 첫 번째 버전을 비교용으로 보관

def cer(ref, hyp):
    # 보완판 정책: ① NFC 정규화 ② 모든 공백 문자 제거 ③ 코드 포인트 단위 비교
    ref = unicodedata.normalize("NFC", ref)
    hyp = unicodedata.normalize("NFC", hyp)
    return score([ch for ch in ref if not ch.isspace()], [ch for ch in hyp if not ch.isspace()])

nfc = "가"                                   # 가 (U+AC00)
nfd = unicodedata.normalize("NFD", nfc)          # 가 (U+1100 + U+1161)
print("NFD 코드 포인트:", [f"U+{ord(c):04X}" for c in nfd], "→ 눈에는 같은 글자")
cases = [("NFC 가 vs NFD 가", nfc, nfd), ("일반 공백 vs NBSP", "hello world", "hello world"), ("탭 vs 공백", "hello\tworld", "hello world")]
for name, a, b in cases:
    print(f"{name:<20} 첫 번째 버전 CER = {cer_v1(a, b)['오류율']:.3f}   보완판 CER = {cer(a, b)['오류율']:.3f}")

NFD 코드 포인트: ['U+1100', 'U+1161'] → 눈에는 같은 글자
NFC 가 vs NFD 가       첫 번째 버전 CER = 2.000   보완판 CER = 0.000
일반 공백 vs NBSP        첫 번째 버전 CER = 0.100   보완판 CER = 0.000
탭 vs 공백              첫 번째 버전 CER = 0.091   보완판 CER = 0.000


첫 번째 버전은 눈에 같은 문자열을 **오류로 셉니다**(`가`는 N=1인데 코드 포인트 2개와 비교해서 2.0). 보완판은 0입니다.

이 보완판은 이후 셀(6절, 9절)의 `cer`에도 그대로 쓰이지만, 그 예시들의 값은 바뀌지 않습니다(공백 문자가 ASCII 공백뿐이므로).

**한계**: 이모지(👨‍👩‍👧처럼 여러 코드 포인트가 한 글자로 보이는 것)는 코드 포인트 단위로 쪼개면 한 글자가 여러 개로 세집니다. 이를 제대로 다루려면 **grapheme cluster** 단위 분리(예: 외부 `regex` 패키지의 `\X`)가 필요합니다 **[일반 지식]**. 영어·한국어·일본어 문장 연습에서는 우선 NFC + 코드 포인트 방식으로 충분하다고 판단합니다.

> **프로젝트 `scoring.py`와의 관계 [실행 확인]**: `scoring.cer()`는 먼저 `normalize()`를 거치고(NFC, 그리고 `\s+`를 한 칸으로 합침), 그 뒤에 `replace(" ", "")`를 합니다. 그래서 탭·NBSP는 이미 일반 공백으로 바뀐 뒤라 위 "ASCII 공백만 지운다"는 문제가 **프로젝트 코드에서는 나타나지 않습니다**. 이 절의 문제는 `normalize` 없이 `cer`만 따로 쓰는 이 노트북의 함수에 해당합니다.

### 4-3. 언어별 토큰화 정책 (보완)

WER는 "단어 단위 오류율"이지만, **무엇을 한 단어(토큰)로 볼지**는 정책입니다. 이 노트북의 `str.split()`은 공백으로 자릅니다.

| 언어 | WER 토큰 | CER 토큰 | 비고 |
|---|---|---|---|
| 영어·프랑스어 | 공백 기준 단어 | 공백 제외 코드 포인트 | 기본 사용 가능 |
| 한국어 | 공백 기준 **어절** | 공백 제외 코드 포인트 | 어절은 형태소 단위 "단어"와 다름 (`날씨가` = `날씨` + 조사 `가`) |
| 일본어 | 공백이 없어 문장 전체가 1토큰이 됨 | 공백 제외 코드 포인트 | 형태소 분석기가 없으면 WER는 의미가 없음 → **CER 중심** |

- 한국어 공백 기준 WER를 "WER는 원래 가혹하다"로 일반화하면 안 됩니다. 토큰화 기준을 바꾸면(예: 형태소) 값이 달라집니다.
- 일본어는 프로젝트 `scoring.wer(..., lang="ja")`도 문장 전체를 **1토큰**으로 봅니다. 즉 `wer`라는 이름으로 나오는 값이 단어 단위 WER가 **아닙니다**(아래 셀에서 `N`이 1임을 확인). 화면에서는 일본어에 CER만 표시하거나, WER를 "참고 값"으로 취급해야 합니다.
- 이 언어별 정책은 오류율을 언어끼리 똑같은 기준으로 비교한다는 뜻이 아닙니다. 언어 특성에 맞춰 평가 단위를 고른 것입니다.

In [8]:
print("한국어(공백 기준 어절):", "오늘 날씨가 좋다".split())
print("일본어(공백 없음)   :", "今日は天気がいい".split(), "← 문장 전체가 토큰 1개 → N = 1")

한국어(공백 기준 어절): ['오늘', '날씨가', '좋다']
일본어(공백 없음)   : ['今日は天気がいい'] ← 문장 전체가 토큰 1개 → N = 1


### 4-4. 경계 사례: 빈 인식·빈 정답·순서 바뀜 (보완)

| 정답 | 인식 | 결과 |
|---|---|---|
| 비어 있지 않음 | 동일 | 오류율 0 |
| 비어 있지 않음 | **빈 문자열** | 정답 토큰이 전부 삭제 → 오류율 1.0 |
| **빈 문자열** | 비어 있지 않음 | 분모 N=0 → 오류율 정의 불가 → 예외 |

STT가 아무것도 인식하지 못했을 때(무음 녹음 등)는 "빈 인식" 경로로 오류율 1.0이 되고, 정답이 비는 경우는 문장 데이터의 오류이므로 예외로 알립니다.

In [9]:
r = wer("hello world", "hello world");  assert r["오류율"] == 0
r = wer("hello world", "");             assert (r["S"], r["D"], r["I"], r["오류율"]) == (0, 2, 0, 1.0)
try:
    wer("", "hello")
except ValueError:
    print("정답이 비어 있으면 ValueError (분모 N = 0) — 정상")
else:
    raise AssertionError("빈 정답은 예외여야 합니다")

# 순서가 바뀐 경우: 단어 집합은 같아도 WER는 0이 아니다 (1-2절 예시)
for a, b in [("I really like this song", "I like this song"), ("the cat sat", "sat the cat")]:
    r = wer(a, b)
    print(f"{a!r} → {b!r}: S={r['S']} D={r['D']} I={r['I']} N={r['N']}  WER={r['오류율']:.3f}")

정답이 비어 있으면 ValueError (분모 N = 0) — 정상
'I really like this song' → 'I like this song': S=0 D=1 I=0 N=5  WER=0.200
'the cat sat' → 'sat the cat': S=0 D=1 I=1 N=3  WER=0.667


## 5. WER는 1을 넘을 수 있다

분모는 **정답 단어 수**인데, 인식 쪽에 **끼어든 단어(삽입)** 는 분자에만 더해집니다. 그래서 삽입이 많으면 1을 넘습니다.
따라서 화면에 `1 − WER` 을 "일치율"로 보여 주면 **음수**가 나올 수 있으므로, 프로젝트에서는 **오류율과 S·D·I 개수**로 표시하기로 했습니다.

In [10]:
r = wer("hello", "hello hello hello")
print(r["S"], r["D"], r["I"], r["N"], "→ WER =", r["오류율"], "  (1 - WER =", 1 - r["오류율"], ")")

0 0 2 1 → WER = 2.0   (1 - WER = -1.0 )


### 5-2. WER는 "정확도(Accuracy)"가 아니라 "정답 길이 대비 오류 횟수의 비율"입니다 (보완)

WER의 분모는 **정답 단어 수**, 분자는 **치환·삭제·삽입 횟수**입니다. 그래서 0 이상이지만 1로 막혀 있지 않습니다.

- 예: 정답 2단어(`a b`)에 단어 3개가 더 끼어들면 $\text{WER} = (0+0+3)/2 = 1.5$ 입니다.
- WER $1.5$ 는 "정답의 150%가 틀렸다"가 아니라 **"정답 길이의 1.5배만큼 편집이 필요했다"** 는 뜻입니다.
- 분류 문제의 Accuracy처럼 $1 - \text{WER}$ 을 정확도로 쓰지 않습니다(음수가 나올 수 있음).

화면에는 다음을 그대로 보여 주기로 합니다: 오류율(WER 또는 CER), 치환 S·삭제 D·삽입 I 개수, 정답 토큰 수 N, 편집 거리 $S+D+I$.

## 6. 단어 단위 vs 글자 단위

| 언어 | 단어 단위(WER)의 문제 | 글자 단위(CER)의 장점 |
|---|---|---|
| 영어 | 단어가 하나만 달라도 단어 전체가 틀림 (`love` vs `loved`) | 글자 하나 차이만 반영 |
| 한국어 | **조사**가 붙어 `날씨가`와 `날씨`가 서로 다른 단어 → WER가 가혹함 | 글자 단위로 차이가 작게 반영 |
| 일본어 | **띄어쓰기가 없어** 단어로 나누려면 별도 도구가 필요함 | 글자 단위(CER)로 바로 계산 가능 |

In [11]:
pairs = [
    ("영어", "i love you", "i loved you"),
    ("한국어", "오늘 날씨가 좋아요", "오늘 날씨 좋아요"),
]
rows = []
for lang, a, b in pairs:
    w, c = wer(a, b), cer(a, b)
    rows.append([lang, a, b, f"{w['거리']}/{w['N']} = {w['오류율']:.3f}", f"{c['거리']}/{c['N']} = {c['오류율']:.3f}"])
pd.DataFrame(rows, columns=["언어", "정답", "인식", "WER", "CER"])

,언어,정답,인식,WER,CER
0,영어,i love you,i loved you,1/3 = 0.333,1/8 = 0.125
1,한국어,오늘 날씨가 좋아요,오늘 날씨 좋아요,1/3 = 0.333,1/8 = 0.125


- `i love you` vs `i loved you`: 단어 하나가 통째로 틀려 WER는 $1/3$ 이지만, 공백을 뺀 글자 8개 중 `d` 하나만 늘었으므로 CER는 $1/8$ 로 더 작습니다(두 문장 모두 같은 값이 되었다는 점에 주의: 단어 수 3개 중 1개 vs 글자 8개 중 1개).
- `날씨가` vs `날씨`: 조사 하나 때문에 WER $= 1/3$ 이지만 CER $= 1/8$ 입니다.
- 일본어는 단어 분할이 필요해서 프로젝트 계획에서도 **CER 중심**으로 두었습니다.

### 6-2. WER·CER는 의미나 문법을 재지 않습니다 (보완)

위 표에서 "한국어는 WER가 가혹하다"는 말은 **공백 기준 어절 WER에 한해서** 맞습니다. 토큰화 기준을 바꾸면 달라집니다(4-3절).

또 WER·CER는 **문자열의 편집 거리**일 뿐이라, 문법적으로 중요한 조사·어미가 빠졌는지, 문장의 뜻이 얼마나 달라졌는지는 판단하지 않습니다.

- 정답 `오늘 날씨가 좋아요` / 인식 `오늘 날씨 좋아요`
  - 어절 WER: `날씨가`→`날씨` 치환 1번 ($1/3$)
  - CER: 조사 `가` 한 글자 삭제 ($1/8$)
  - 같은 차이를 **다른 단위**로 표현한 것이며, 조사 누락의 문법적 의미는 숫자로 드러나지 않습니다.

따라서 오류율만 보지 말고 **실제 인식 문장**을 함께 확인합니다. 일본어는 히라가나/가타카나, 한자/히라가나 표기, 전각/반각 같은 정규화 정책이 따로 필요하므로 이 프로젝트에서 실제로 지원하기 전까지는 "언어별 특징"으로만 남겨 둡니다.

## 7. 정규화가 점수를 바꾼다

STT는 대소문자·구두점을 붙여서 내놓기도 합니다. 정답과 **글자가 같아도 형식이 다르면 오류로 센다**는 문제가 생기므로, 비교 전에 **같은 규칙으로 정리(정규화)** 합니다.

In [12]:
ref_raw = "Hello, World!"
hyp_raw = "hello world"

before = wer(ref_raw, hyp_raw)
print("정규화 전: WER =", before["오류율"], f"(S={before['S']} D={before['D']} I={before['I']})")

정규화 전: WER = 1.0 (S=2 D=0 I=0)


In [13]:
import re, unicodedata

def normalize_en(text):
    """영어 정규화 예시: NFC → 소문자 → 구두점 제거(아포스트로피는 유지) → 연속 공백 정리"""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s']", "", text)
    return re.sub(r"\s+", " ", text).strip()

after = wer(normalize_en(ref_raw), normalize_en(hyp_raw))
print("정규화 후: WER =", after["오류율"])
print(normalize_en("Don't  stop,   believin'!"))

정규화 후: WER = 0.0
don't stop believin'


- 같은 소리를 인식했는데 정규화 전에는 WER $=1.0$, 후에는 $0$ 입니다. **점수가 아니라 규칙이 결과를 바꾼 것**이므로, 프로젝트에서는 규칙을 고정하고 문서화하기로 했습니다.
- 위 규칙은 **예시**입니다. 축약형(`don't`)·숫자 표기를 어떻게 다룰지는 프로젝트 1일차에 결정합니다.
- 참고(검토 문서 7절에서 확인한 값): 수업 제공 한국어 WAV 5개를 Whisper tiny로 인식한 합산 결과는 **구두점을 제거하면 WER 0.060 / CER 0.034, 그대로 두면 WER 0.200** 이었습니다. 이 노트북에서는 다시 실행하지 않았습니다.

### 7-2. 정규화는 "점수를 좋게 만드는 일"이 아니라 "평가 정책"입니다 (보완)

정규화는 무엇을 오류로 세고 무엇을 무시할지 정하는 **평가 규칙**입니다. 위 예에서 WER가 1.0 → 0이 된 것은 인식이 좋아져서가 아니라, 대소문자와 구두점 차이를 **무시하기로** 했기 때문입니다.

| 항목 | 예시 | 정해야 할 것 |
|---|---|---|
| 대소문자 | Hello / hello | 소문자로 통일? |
| 구두점 | hello! / hello | 구두점 차이를 평가? |
| 아포스트로피 | don't / dont | 유지? 제거? |
| 아포스트로피 모양 | don't / don’t | 곧은 것(U+0027)과 둥근 것(U+2019)을 통일? |
| 축약형 | we'll / we will | 같게 볼 것인가? (프로젝트 이슈 #3, 미확정) |
| 숫자 | twenty / 20 | 숫자 표기를 통일? (미확정) |
| 밑줄·기타 | `\w`에 `_`가 들어감 | 남길지 정하기 |

규칙은 정답과 인식 결과에 **똑같이** 적용해야 합니다. 그리고 **원본 문장은 보존**하고 정규화된 문장은 따로 만듭니다(정규화만 저장하면 실제 STT 출력이 무엇이었는지 나중에 알 수 없습니다). 결과를 기록할 때는 정규화 규칙(또는 버전)도 같이 적어 둡니다.

In [14]:
import re, unicodedata

def normalize_en_v2(text):
    # 영어 전용 정규화(보완판): NFC → 소문자 → 둥근 아포스트로피(’)를 곧은 것(')으로 → 영문자·숫자·공백·'만 남김 → 공백 정리
    text = unicodedata.normalize("NFC", text).lower()
    text = text.replace("’", "'")
    text = re.sub(r"[^a-z0-9\s']", "", text)
    return re.sub(r"\s+", " ", text).strip()

print(repr(normalize_en("Don’t stop")), "← 이전 normalize_en: 둥근 아포스트로피는 [^\\w\\s']에 걸려 지워짐")
print(repr(normalize_en_v2("Don’t stop")), "← 보완판: 곧은 것으로 통일")
print(repr(normalize_en_v2("Café")), "← 영어 전용이라 é가 지워짐. 프랑스어는 별도 함수가 필요")

'dont stop' ← 이전 normalize_en: 둥근 아포스트로피는 [^\w\s']에 걸려 지워짐
"don't stop" ← 보완판: 곧은 것으로 통일
'caf' ← 영어 전용이라 é가 지워짐. 프랑스어는 별도 함수가 필요


**확인된 함정 — 작은따옴표로 감싼 단어 [실행 확인]**

아포스트로피(`'`)를 유지하는 규칙은 `don't`에는 맞지만, `'hello'`처럼 **따옴표로 감싼 단어**의 바깥 `'`도 남깁니다. 곧은 따옴표뿐 아니라 `‘hello’`도 `’`가 `'`로 바뀌어 `hello'`가 됩니다. 이 때문에 `'hello' world`와 `hello world`는 다른 문장으로 계산됩니다(WER 0.5). 반대로 `believin'`처럼 **끝의 `'`가 의미 있는** 표기도 있어서(발음 생략 표기), 단어 끝 `'`를 무조건 지우는 것도 답이 아닙니다. 이 경계는 프로젝트의 정규화 방침(이슈 #3)에서 결정할 항목입니다.

In [15]:
for a, b in [("'hello' world", "hello world"), ("Don't stop believin'", "don't stop believin")]:
    r = wer(normalize_en_v2(a), normalize_en_v2(b))
    print(f"{a!r} vs {b!r}: S={r['S']} D={r['D']} I={r['I']} → WER {r['오류율']:.3f}")

"'hello' world" vs 'hello world': S=1 D=0 I=0 → WER 0.500
"Don't stop believin'" vs "don't stop believin": S=1 D=0 I=0 → WER 0.333


## 8. `difflib`로 센 오류는 편집 거리와 다를 수 있다

파이썬 표준 라이브러리 `difflib.SequenceMatcher`는 두 열의 **눈에 띄는 차이**를 보여 주는 도구이고, **최소 편집 경로를 보장하는 알고리즘은 아닙니다** **[일반 지식 + 아래 실험으로 확인]**. 그래서 `difflib` 결과로 오류 개수를 세면 WER 계산(편집 거리)과 **어긋날 수 있습니다.**

In [16]:
import difflib, random

def difflib_cost(a, b):
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    cost = 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        cost += max(i2 - i1, j2 - j1)       # 치환은 긴 쪽 길이, 삭제·삽입은 해당 길이
    return cost

a = ['b', 'c', 'b', 'a', 'c', 'a', 'a']
b = ['c', 'b', 'c', 'c', 'c', 'a']
print("difflib로 센 오류:", difflib_cost(a, b), " / 편집 거리:", edit_matrix(a, b)[-1][-1])

# 무작위 3글자 열 20000쌍으로 비교 (아주 반복적인 열이라 극단적인 경우)
random.seed(0)
mismatch = 0
for _ in range(20000):
    x = [random.choice("abc") for _ in range(random.randint(3, 8))]
    y = [random.choice("abc") for _ in range(random.randint(3, 8))]
    if difflib_cost(x, y) != edit_matrix(x, y)[-1][-1]:
        mismatch += 1
print("어긋난 쌍:", mismatch, "/ 20000")

difflib로 센 오류: 4  / 편집 거리: 3
어긋난 쌍: 8429 / 20000


- 위 실험은 **3글자만 쓰는 무작위 열**이라 극단적인 경우입니다. 실제 문장에서 얼마나 자주 어긋나는지는 **측정하지 않았습니다.**
- 그래도 "점수(편집 거리)와 화면의 오류 표시가 **같은 경로**여야 한다"는 이유로, 프로젝트에서는 **역추적으로 얻은 치환·삭제·삽입**을 표시에 쓰고 `difflib`은 보조로만 씁니다.

### 8-2. `difflib`와 Levenshtein은 "목적"이 다릅니다 (보완)

두 도구 모두 두 열의 차이를 보여 주지만 목적과 알고리즘이 다릅니다.

| | Levenshtein(편집 거리) | `difflib.SequenceMatcher` |
|---|---|---|
| 목적 | 치환·삭제·삽입의 **최소 횟수** | 사람이 읽기 좋은 **같은 부분 / 다른 부분** 표시 |
| 방법 | 동적 계획법 | 가장 긴 공통 블록을 찾고 양쪽으로 나눠 반복 |
| 최소 보장 | 있음 | **없음** |
| WER·CER 기준 | 사용 | 사용하지 않음 |

- 두 결과가 다르다고 어느 한쪽이 "버그"인 것이 아닙니다. 서로 다른 기준으로 열을 맞춘 것입니다.
- 위 8절의 `difflib_cost()`는 "치환은 긴 쪽 길이, 삭제·삽입은 해당 길이"로 센 **이 노트북이 정한 대략적인 환산 규칙**입니다. Levenshtein 공식이 아니므로, 어긋난 쌍의 개수(20000쌍 중 8429쌍)는 이 환산 규칙에 따른 값이며 `difflib` 일반의 오류율이 아닙니다 **[추정]**.
- 프로젝트 역할 분담: 점수는 Levenshtein 기반 S·D·I, 틀린 부분 표시도 **같은 역추적 결과**, `difflib`은 동률일 때 단어 닮음 점수(`SequenceMatcher.ratio()`)에만 보조로 씁니다.

## 9. 연습 문제

**먼저 종이에 직접** $S,\ D,\ I,\ N$ 과 WER를 계산한 뒤, 아래 확인 셀의 결과와 비교해 보세요.

| 문제 | 정답(reference) | 인식(hypothesis) |
|---|---|---|
| ① | i want to explore the universe | i want explore a universe |
| ② | she sells sea shells | she sells sea shells by the shore |
| ③ | 오늘 날씨가 좋아요 | 오늘 날씨 좋아요 (CER도 계산) |

In [17]:
problems = [
    ("①", "i want to explore the universe", "i want explore a universe", "WER"),
    ("②", "she sells sea shells", "she sells sea shells by the shore", "WER"),
    ("③", "오늘 날씨가 좋아요", "오늘 날씨 좋아요", "CER"),
]
for no, a, b, kind in problems:
    r = (wer if kind == "WER" else cer)(a, b)
    print(f"{no} {kind}: S={r['S']} D={r['D']} I={r['I']} N={r['N']} → {r['거리']}/{r['N']} = {r['오류율']:.3f}")

① WER: S=1 D=1 I=0 N=6 → 2/6 = 0.333
② WER: S=0 D=0 I=3 N=4 → 3/4 = 0.750
③ CER: S=0 D=1 I=0 N=8 → 1/8 = 0.125


**풀이 확인**

- ① 정답 6단어. `to` 삭제 1, `the`→`a` 치환 1 → $(1+1)/6 = 0.333$
- ② 정답 4단어. 인식에 `by the shore` 3단어가 **삽입** → $3/4 = 0.75$ (삽입은 분자에만 더해짐)
- ③ 공백을 뺀 정답은 8글자, 조사 `가` 하나가 삭제 → $1/8 = 0.125$

(세 값이 위 출력과 다르면 어디서 갈렸는지 표와 역추적을 다시 따라가 봅니다.)

## 10. 여러 문장의 WER/CER를 어떻게 합칠까 — Micro와 Macro (보완)

프로젝트에서는 문장 하나가 아니라 **여러 문장·여러 번의 연습**을 모아 보게 됩니다(10문장 × 3회 검증, 기록 그래프). 이때 합치는 방법이 두 가지이고 값이 다릅니다.

| 문장 | 정답 단어 수 N | 편집 오류 수 | 문장별 WER |
|---|---|---|---|
| A | 2 | 1 | 0.5 |
| B | 10 | 1 | 0.1 |

- **Macro 평균(문장별 평균)**: 문장마다 WER를 구해 평균합니다. $(0.5 + 0.1)/2 = 0.3$. 모든 문장이 **같은 비중**입니다.
- **Micro 평균(오류 합산)**: 오류 횟수와 정답 단어 수를 각각 합친 뒤 나눕니다. $(1+1)/(2+10) \approx 0.167$. 단어가 많은 문장의 **비중이 큽니다**.

일반적으로 코퍼스 전체의 WER는 Micro 방식(전체 오류 ÷ 전체 정답 단어 수)으로 보고합니다 **[일반 지식]**. 짧은 문장 하나가 평균을 크게 흔드는 것을 막아 주기 때문입니다. 10문장 × 3회 검증에서는 **Micro를 기본 지표, Macro를 보조 지표**로 함께 적고, 문장별 값도 같이 두는 것이 좋습니다.

In [18]:
def aggregate_micro(results):
    # score()가 돌려준 dict 여러 개를 합산해 Micro 오류율을 계산한다 (정답 토큰 합이 0이면 계산 불가)
    S = sum(r["S"] for r in results); D = sum(r["D"] for r in results)
    I = sum(r["I"] for r in results); N = sum(r["N"] for r in results)
    if N == 0:
        raise ValueError("전체 정답 토큰 수가 0이라 오류율을 계산할 수 없습니다")
    return {"S": S, "D": D, "I": I, "N": N, "거리": S + D + I, "오류율": (S + D + I) / N}

def aggregate_macro(results):
    # 문장별 오류율의 단순 평균 (N이 0인 문장은 score()가 이미 예외를 내므로 여기에 오지 않는다)
    return sum(r["오류율"] for r in results) / len(results)

results = [
    wer("a b", "a"),
    wer("one two three four five six seven eight nine ten", "one two three four five six seven eight nine"),
]
micro, macro = aggregate_micro(results), aggregate_macro(results)
print("문장별 WER :", [round(r["오류율"], 3) for r in results])
print(f"Macro 평균 : {macro:.3f}")
print(f"Micro 평균 : {micro['거리']}/{micro['N']} = {micro['오류율']:.3f}")
assert abs(macro - 0.3) < 1e-9 and abs(micro["오류율"] - 2 / 12) < 1e-9

문장별 WER : [0.5, 0.1]
Macro 평균 : 0.300
Micro 평균 : 2/12 = 0.167


## 11. 테스트와 경계 조건 (보완)

WER/CER는 알고리즘 자체는 단순하지만 입력의 경계(빈 입력, 길이 차이, 유니코드)에서 틀리기 쉽습니다. 프로젝트로 옮길 때는 아래 **불변 조건(invariant)** 을 자동 테스트로 두면 `scoring.py`를 고쳐도 핵심 계산이 깨지지 않았는지 확인할 수 있습니다.

1. $S + D + I$ = 편집 거리 (역추적이 표와 일치)
2. 정답과 인식이 같으면 오류율 0
3. 인식이 빈 문자열이면 정답 토큰이 전부 삭제 (오류율 1.0)
4. 정답이 빈 문자열이면 계산 불가(예외)
5. 삽입이 많으면 오류율이 1을 넘을 수 있음
6. 편집 거리는 $|N - M|$ 이상이고 $\max(N, M)$ 이하
7. 편집 거리는 정답과 인식을 **바꿔도 같음** (대칭). 단, S와 D·I는 서로 바뀝니다.
8. 정규화 뒤 눈에 같은 문자열은 같은 토큰 열

In [19]:
import random

# 개별 사례
r = wer("hello world", "hello word");   assert (r["S"], r["D"], r["I"]) == (1, 0, 0)    # 치환
r = wer("hello world", "hello");        assert (r["S"], r["D"], r["I"]) == (0, 1, 0)    # 삭제
r = wer("hello", "hello world");        assert (r["S"], r["D"], r["I"]) == (0, 0, 1)    # 삽입
r = wer("hello", "hello hello hello");  assert (r["I"], r["오류율"]) == (2, 2.0)           # WER > 1
assert cer(nfc, nfd)["오류율"] == 0                                                        # NFC/NFD 같은 글자

# 무작위 불변 조건 (500쌍)
random.seed(0)
for _ in range(500):
    x = [random.choice("abc") for _ in range(random.randint(1, 8))]
    y = [random.choice("abc") for _ in range(random.randint(0, 8))]
    r = score(x, y)
    assert r["S"] + r["D"] + r["I"] == r["거리"]                            # 1
    assert abs(len(x) - len(y)) <= r["거리"] <= max(len(x), len(y))        # 6
    if y:
        assert edit_matrix(x, y)[-1][-1] == edit_matrix(y, x)[-1][-1]      # 7 (대칭)
print("개별 사례 + 무작위 500쌍 불변 조건 통과")

개별 사례 + 무작위 500쌍 불변 조건 통과


## 12. 프로젝트 `scoring.py`와의 차이 (보완)

이 노트북의 함수는 학습용으로 단순하게 만든 것이고, 프로젝트의 `scoring.py`는 여기에 아래를 더했습니다.

| | 이 노트북 | `scoring.py` |
|---|---|---|
| 정규화 | 별도 함수를 직접 부름 | `wer`/`cer`가 안에서 `normalize(text, lang)`을 호출 (언어별 규칙) |
| 결과 dict 키 | `"오류율"`, `"거리"` (한글) | `"rate"`, `"distance"` (영문) |
| 역추적 동률 | `if` 순서만 (일치/치환 → 삭제 → 삽입) | `_tie_scores()`로 **닮은 단어끼리 짝짓는 경로** 우선, 그래도 같으면 `if` 순서 |
| 표시 | 직접 출력 | `format_diff(ops)`가 `[정답→인식]`, `[-삭제-]`, `[+삽입+]` 표시 |
| 한 번에 평가 | 없음 | `evaluate()`가 WER·CER와 표시용 쪽(`main`)을 함께 반환 |

아래 셀은 3-2절의 `we'll sing together` 예시를 두 구현으로 나란히 돌립니다(이 셀은 프로젝트 폴더의 `scoring.py`를 불러오므로 프로젝트 안에서 열었을 때만 실행됩니다).

In [20]:
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "scoring.py").exists() and _root != _root.parent:   # 이 노트북은 experiments/ 아래 폴더에 있으므로
    _root = _root.parent                                              # 위로 올라가며 프로젝트 루트(scoring.py가 있는 곳)를 찾는다
sys.path.insert(0, str(_root))
import scoring

a, b = "We'll sing together", "We are seeing together"
mine = wer(normalize_en_v2(a), normalize_en_v2(b))     # 이 노트북 버전 (미리 정규화해서 넣음)
proj = scoring.wer(a, b)                               # 프로젝트 버전 (안에서 정규화)

print("노트북 :", scoring.format_diff(mine["ops"]), f"  S={mine['S']} D={mine['D']} I={mine['I']}  WER={mine['오류율']:.3f}")
print("프로젝트:", scoring.format_diff(proj["ops"]), f"  S={proj['S']} D={proj['D']} I={proj['I']}  WER={proj['rate']:.3f}")
assert mine["거리"] == proj["distance"]               # 경로는 달라도 거리(오류율)는 같다

노트북 : [+we+] [we'll→are] [sing→seeing] together   S=2 D=0 I=1  WER=1.000
프로젝트: [we'll→we] [+are+] [sing→seeing] together   S=2 D=0 I=1  WER=1.000


## 13. 정리와 프로젝트 연결

**핵심 정리**

- $\text{WER} = (S+D+I)/N$, $N$ 은 **정답**의 단어 수. CER는 글자 단위로 같은 계산.
- $S+D+I$ 는 **편집 거리**이고, 표(DP)로 구하고 **역추적**으로 어디가 틀렸는지 얻는다.
- 삽입이 많으면 **1을 넘을 수 있다** → "일치율 = 1 − WER"로 쓰지 않는다.
- 정답이 비면 계산 불가 → 오류로 처리.
- 정규화 규칙(대소문자·구두점·공백·유니코드 등)에 따라 점수가 크게 달라진다 → 규칙을 고정하고 기록.
- **주의**: WER·CER는 "인식 결과가 정답 문장과 얼마나 다른가"이지 **발음 점수가 아니다.** STT는 문맥으로 틀린 발음을 올바른 문장으로 고쳐 적을 수 있다.

**프로젝트 `scoring.py` 로 옮길 때의 목록**

| 함수 | 역할 |
|---|---|
| `normalize(text, lang)` | 언어별 정규화 |
| `edit_matrix`, `backtrace` | 편집 거리 표와 역추적 |
| `score(ref_tokens, hyp_tokens)` | S·D·I·N·오류율 반환, 빈 정답 처리 |
| `wer`, `cer` | 단어/글자 단위 호출 |
| 테스트 | 손계산 5건, 경계 사례(빈 정답, 동일 문장, WER > 1), `S+D+I == 거리` 검산 |
- **보완 (2026-11-02)**:
  - WER는 단어 집합이 아니라 **순서가 있는 열**의 정렬이다. 최적 경로는 여러 개일 수 있고, 거리는 같아도 S·D·I와 표시가 달라질 수 있다(3-2).
  - CER는 계산 전에 NFC로 맞추고 모든 공백 문자를 제거한다(4-2). 일본어는 `wer`가 문장 전체 1토큰이라 CER 중심(4-3).
  - 여러 문장은 **Micro(오류 합산)를 기본**, Macro(문장별 평균)를 보조로 함께 적는다(10).
  - 따옴표로 감싼 단어(`'hello'`)와 `believin'` 같은 끝 아포스트로피는 정규화 방침(#3)에서 정해야 할 경계다(7-2).